# 사투리 번역기 — AI Hub 방언 데이터 (부분 다운로드 → 전처리)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysb2152/translate/blob/main/notebooks/aihub_colab.ipynb)

> ⚠️ **지오블록 주의 (중요)**: AI Hub는 **해외 IP에서의 다운로드를 차단**한다(`해외에서의 데이터 다운로드를 제한`). Colab 서버는 해외라 **아래 다운로드 셀(3)이 막힌다.** → **다운로드는 한국 IP(집 PC)** 에서 하고(브라우저 또는 로컬 aihubshell), 결과(전처리 산출물 또는 음성 클립)만 Google Drive로 옮긴 뒤 이 노트북의 **전처리·학습 셀**을 쓴다. 다운로드 셀은 한국에 있는 서버/PC에서만 동작한다.

**용량 문제**: AI Hub 방언 데이터는 지역당 수백 GB라 전량은 불필요하고 로컬(34GB)엔 안 들어간다. `aihubshell`로 **필요한 청크(filekey)만 부분 다운로드**한다.

**흐름**: (한국 IP에서) 부분 다운로드 → 압축 해제 → 저장소 clone → `data/preprocess.py`로 STT 매니페스트 + (방언→표준) 문장쌍 생성 → Google Drive에 저장.

> 학습(Whisper 파인튜닝)은 데이터가 준비된 다음 셀/노트북에서 이어간다(B-9).

## 0. 준비물

1. **AI Hub 계정** + 받으려는 데이터셋의 **활용 신청 승인**(마이페이지에서 상태 확인).
2. **API 키**: AI Hub 마이페이지 → *API 키 발급*에서 발급받아 아래 프롬프트에 입력.
3. 런타임: **런타임 → 런타임 유형 변경 → GPU** (전처리는 CPU로 충분하지만, 이어서 학습하려면 GPU).

> 키는 노트북에 하드코딩하지 말고 실행할 때만 입력. 공유 시 반드시 지운다.

In [ ]:
# (선택) Google Drive 마운트 — 전처리 산출물을 세션이 꺼져도 남기려면 사용
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = '/content/drive/MyDrive/saturi/processed'
else:
    OUT_ROOT = '/content/processed'
print('산출물 경로:', OUT_ROOT)

In [ ]:
# 설정
import getpass, os

# 데이터셋 키(dataSetSn): 경상도=119, 전라도=120, 충청도=122, 강원도=121, 제주도=123 ...
DATASET_KEY = 119

# API 키(입력 프롬프트로 받음 — 노트북에 저장 안 함)
AIHUB_APIKEY = getpass.getpass('AI Hub API 키 입력: ').strip()

# 작업 폴더(로컬 ephemeral — 빠름). 원본은 여기 받고, 산출물만 Drive로.
RAW_DIR = '/content/aihub_raw'
os.makedirs(RAW_DIR, exist_ok=True)
print('dataset', DATASET_KEY, '| raw', RAW_DIR, '| out', OUT_ROOT)

In [ ]:
# 1) aihubshell 설치
!curl -sL -o /usr/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do
!chmod +x /usr/bin/aihubshell
!aihubshell -help 2>/dev/null | head -20 || echo '설치 확인: -help 출력이 없으면 URL/네트워크 확인'

In [ ]:
# 2) 파일 목록(filekey)과 용량 확인 → 여기서 받을 청크를 고른다
#    출력의 각 파일 뒤 숫자가 filekey. 라벨(작음) + 그에 대응하는 원천(음성) 소량만.
!aihubshell -mode l -datasetkey {DATASET_KEY} -aihubapikey "{AIHUB_APIKEY}"

## 3. 부분 다운로드

아래 `FILEKEYS`에 **받을 파일 번호만** 넣는다(그 줄만 수정). 여러 개면 콤마.

**경상도(119) 권장 순서**
- 1단계(지금): 라벨만 `572701` (308MB) → 방언→표준 문장쌍 전량 + 실제 라벨 구조 확인
- 2단계(나중): 음성 한 덩어리 `572702` (경상도_1, 28GB) → STT 매니페스트

> 팁: 디스크가 빠듯하면 zip 해제 직후 zip 삭제, 전처리 후 원본 삭제(맨 아래 증분 셀 참고).

In [ ]:
# 3) 고른 청크만 다운로드 — 이 아래 FILEKEYS 값만 바꾸면 된다
FILEKEYS = '572701'          # 경상도 라벨(Training) 308MB. 여러 개면 '572701,572702'

assert FILEKEYS.strip(), 'FILEKEYS 에 filekey 번호를 넣으세요(위 목록 셀 참고).'
!cd {RAW_DIR} && aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {FILEKEYS} -aihubapikey "{AIHUB_APIKEY}"
!echo '--- 받은 파일 ---' && find {RAW_DIR} -maxdepth 3 -type f | head -40
!echo '--- 사용량 ---' && du -sh {RAW_DIR}

In [ ]:
# 4) 압축 해제 (AI Hub는 .zip 로 내려옴; 여러 파트로 쪼개진 경우 병합 후 해제)
!apt-get -qq install -y p7zip-full > /dev/null 2>&1
import glob, os, subprocess
for z in glob.glob(f'{RAW_DIR}/**/*.zip', recursive=True):
    print('unzip:', z)
    subprocess.run(['7z', 'x', '-y', z, f'-o{os.path.dirname(z)}'],
                   stdout=subprocess.DEVNULL)
# 디스크 아끼려면 해제 후 zip 삭제:  import glob,os; [os.remove(z) for z in glob.glob(f'{RAW_DIR}/**/*.zip', recursive=True)]
!echo '--- 해제 후 구조 ---' && find {RAW_DIR} -maxdepth 5 -type d | head -50
!echo '--- json/wav 개수 ---' && echo "json: $(find {RAW_DIR} -name '*.json' | wc -l)" && echo "wav: $(find {RAW_DIR} -iname '*.wav' | wc -l)"

In [ ]:
# 4-b) 라벨 JSON 실제 구조 확인 (전처리 FIELDS 교정용) — 한 개만 열어 본다
import glob, json
js = glob.glob(f'{RAW_DIR}/**/*.json', recursive=True)
print('json 파일 수:', len(js))
if js:
    obj = json.load(open(js[0], encoding='utf-8'))
    print('최상위 키:', list(obj.keys()))
    print(json.dumps(obj, ensure_ascii=False, indent=2)[:1500])

In [ ]:
# 5) 전처리 스크립트 가져오기(저장소 clone) 후 실행
![ -d translate ] && (cd translate && git pull -q) || git clone -q https://github.com/ysb2152/translate
# raw 구조가 세션 오디오+start/end 면 기본(슬라이스), 이미 발화 단위면 --no-slice 추가
!cd translate && python data/preprocess.py --raw {RAW_DIR} --out {OUT_ROOT} --val-ratio 0.05

In [ ]:
# 6) 산출물 확인
import json, itertools, os
print('=== stats ===')
print(open(f'{OUT_ROOT}/stats.json', encoding='utf-8').read())
for name, path in [('STT', f'{OUT_ROOT}/stt/train.jsonl'), ('MT', f'{OUT_ROOT}/mt/train.jsonl')]:
    print(f'\n=== {name} 미리보기 ===')
    if os.path.exists(path):
        for line in itertools.islice(open(path, encoding='utf-8'), 3):
            print(line.strip())

## 7. 디스크가 빠듯하면: 청크 단위 증분 처리

원본을 다 쌓지 말고 **한 청크 받기 → 전처리 → 원본 삭제 → 다음**을 반복(전처리가 작은 클립+매니페스트만 남김). 매니페스트는 실행마다 덮어쓰므로, 청크별로 `--out .../part_1`, `part_2` … 로 나눠 만든 뒤 jsonl 을 이어붙인다(클립 경로가 절대경로라 병합 안전).

```python
for i, fk in enumerate(['572702', '572706']):   # 받을 음성 청크들
    !rm -rf {RAW_DIR}/*
    !cd {RAW_DIR} && aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {fk} -aihubapikey "{AIHUB_APIKEY}"
    # (셀 4 압축해제 실행)
    !cd translate && python data/preprocess.py --raw {RAW_DIR} --out {OUT_ROOT}/part_{i} --val-ratio 0.05
```

## 다음 단계 (B-9) — Whisper 파인튜닝

데이터(`{OUT_ROOT}/stt/*.jsonl`)가 준비되면:
1. **baseline 측정**: 표준 `whisper-base/small`로 val 셋 CER/WER 측정(사투리에서 얼마나 틀리는지).
2. **파인튜닝**: HuggingFace `transformers`로 Whisper를 방언 데이터에 파인튜닝.
3. **개선 측정**: 같은 val 셋에서 CER/WER 재측정 → 개선폭이 핵심 포트폴리오 지표.
4. **서빙 연결**: CTranslate2로 변환 후 백엔드 `WHISPER_MODEL_DIR`에 연결.

이 부분은 데이터가 실제로 준비된 뒤 별도 셀로 이어간다.